# YOLO
This notebook is used to train and test the model

## Imports

In [1]:
import sys
import cv2
from pathlib import Path
# from google.colab.patches import cv2_imshow

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [2]:
# Set execution root in the project root
sys.path.insert(0, str(Path.cwd().parent))

In [3]:
from src.utils.config.config import Config
from src.detector.yolo_detector import YOLODetector

## Config

In [4]:
config_loader = Config()
cfg = config_loader.load_config()

yolo_weights = cfg.paths.models / (config_loader.get("model", "yolo.pretrained"))
yolo_imgsz = config_loader.get("model", "yolo.imgsz")

dataset_yaml = cfg.project_root / cfg.paths.dataset / "CafeV1" / "yolo" / "cafe_v1.yaml"

## Init YOLO

In [5]:
# Passing the .pt weights path here loads YOLOv8m with the pretrained weights. 
detector = YOLODetector(yolo_weights, cfg.project_root)

YOLOv8m summary: 169 layers, 25,902,640 parameters, 0 gradients, 79.3 GFLOPs


## Train

In [6]:
results = detector.train(dataset_yaml, yolo_imgsz)

New https://pypi.org/project/ultralytics/8.4.56 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.46  Python-3.10.11 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 3070, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Research\Dataset\data\processed\CafeV1\yolo\cafe_v1.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Research\Prototype

KeyboardInterrupt: 

## Patch checkpoint and resume

Shorten total epochs so `close_mosaic` triggers at epoch 30 instead of 90, and keep patience generous enough that the mosaic-off phase has room to find a new best.

In [6]:
ckpt_path = cfg.project_root / "runs" / "train" / "weights" / "last.pt"

In [ ]:
import torch

ckpt = torch.load(ckpt_path, weights_only=False)

# close_mosaic stays at 10 -> mosaic off at epoch (epochs - 10) = 30
ckpt["train_args"]["epochs"] = 40
ckpt["train_args"]["patience"] = 35

torch.save(ckpt, ckpt_path)
print("Patched:", ckpt["train_args"]["epochs"], "epochs,", ckpt["train_args"]["patience"], "patience")

Patched: 40 epochs, 35 patience


In [7]:
# resume=True reuses the patched train_args from last.pt
# (optimizer state, EMA, and LR schedule continue from where they were)
from ultralytics import YOLO

resume_model = YOLO(ckpt_path)
resume_results = resume_model.train(resume=True)

New https://pypi.org/project/ultralytics/8.4.56 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.46  Python-3.10.11 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 3070, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Research\Dataset\data\processed\CafeV1\yolo\cafe_v1.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Research\Prototype\

## Test

In [ ]:
frame = cv2.imread("test.jpg")
results = detector.predict(frame)

print(results)

In [ ]:
# Process results list
for result in results[:10]:
    boxes = result.boxes  # Boxes object for bounding box outputs
    # print(boxes.data.tolist()) # x1, y1, x2, y2, conf score, class id
    masks = result.masks  # Masks object for segmentation masks outputs
    keypoints = result.keypoints  # Keypoints object for pose outputs
    probs = result.probs  # Probs object for classification outputs
    obb = result.obb  # Oriented boxes object for OBB outputs
    result.show()  # display to screen
    # result.save(filename="result.jpg")  # save to disk